In [ ]:
'''
새로운 코드를 작성하고 싶어. 거시적으로 보면 './3) execute' 폴더의 csv들을 이용해 problem_solutions.csv 및 problem_testcases.csv에서 특정 조건에 해당하는 것만 남기고 새로운 열을 추가해서 새로운 validated_problem_solutions.csv 및 validated_problem_testcases.csv를 만드는 작업이야. 아래를 읽어보고, 일단 이해가 안 되는 부분은 질문부터 해줘

1. problem_solutions.csv를 읽는다. 이때, 이 파일은 크기가 크니 배치 단위 10000으로 읽는다. 각 행의 problem_id, solution_order 값에 대해 './3) execute/{problem_id}-{solution_order}.csv'가 있는지 검사한다. 없으면 해당 행의 validation 값을 false로 수정하고 넘기고, 있으면 해당 csv를 읽어서 validation 열값들을 검사한다. 만약 값들 중 하나라도 false가 있다면 역시나 problem_solutions.csv의 해당 행의 validation 값을 false로 수정하고 넘기고, 모두 true라면 true로 수정하며 problem_testcases.csv의 problem_id에 해당하는 모든 testcase 행들의 validation={solution_order} 값으로 설정한다. 모두 true인게 발견된 경우 다음 해당 problem_id의 다음 solution_order들은 검사할 필요가 없으므로 다음 것들의 validation은 false로 지정하고 다음 문제로 넘어간다. 이때, problem_testcases.csv 또한 파일 크기가 크니 배치 단위 10000으로 읽는다. 이런 식으로 problem_solutions.csv의 각 행을 처리할 때마다 print해서 중간 진행상황을 출력한다(어느 problem_id 및 solution_order에 대해 validation=true/false 중 무엇을 부여했는지 보고)
2. 위와 같은 방식으로 끝까지 다 처리하면 problem_solutions.csv에서 각 문제의 solution_order들 중 최대 하나의 validation만 true가 되어 있을 것이고, problem_testcases.csv에서 각 문제의 테스트케이스들의 validation은 모두 null이거나 모두 특정 숫자(solution_order)로 되어 있을 것이다.
3. 이 상황에서 각각의 csv를 validating_problem_solutions.csv, validating_problem_testcases.csv로 저장한다. 중간 저장용 파일이다.
4. 이제 각각의 csv에서 validation의 값이 null 및 false가 아닌 행들만 추려서 저장할 것이다. problem_solutions.csv에서는 true여야 하고, problem_testcases.csv에서는 숫자여야 한다. 각각의 기준으로 걸러진 테이블을 validated_problem_solutions.csv, validated_problem_testcases.csv로 저장한다.
'''

In [1]:
import os
import pandas as pd

# 설정값
EXECUTE_DIR = './3) execute'
CHUNK_SIZE = 10000

# 파일 경로 설정 (입력)
PROBLEMS_FILE = 'problems.csv'
SOLUTIONS_FILE = 'problem_solutions.csv'
TESTCASES_FILE = 'problem_testcases.csv'

# 파일 경로 설정 (출력 - 최종 알짜배기 파일)
VALIDATED_PROB_FILE = 'validated_problems.csv'
VALIDATED_TC_FILE = 'validated_problem_testcases.csv'

# 상태 저장을 위한 딕셔너리 및 집합
successful_orders = {} # { problem_id : 통과한_solution_order }
successful_codes = {}  # { problem_id : 통과한_solution_code }
passed_problem_ids = set() # 한 번이라도 통과한 문제가 있으면 기록 (조기 종료 및 필터링용)

print("🚀 [Step 1] problem_solutions.csv 검사 시작 (정답 솔루션 코드 확보)")

# --- Step 1: problem_solutions.csv 청크 단위 읽기 및 정답 탐색 ---
chunk_iter_sol = pd.read_csv(SOLUTIONS_FILE, chunksize=CHUNK_SIZE)

for chunk_idx, df_sol_chunk in enumerate(chunk_iter_sol):
    print(f"  > [Sol] Chunk {chunk_idx + 1} 처리 중...")
    
    for idx, row in df_sol_chunk.iterrows():
        prob_id = row['problem_id']
        sol_order = row['solution_order']
        sol_code = row['solution']
        
        # 이미 정답을 찾은 문제라면 하위 솔루션은 더 볼 필요 없이 패스
        if prob_id in passed_problem_ids:
            continue
            
        csv_path = f"{EXECUTE_DIR}/{prob_id}-{sol_order}.csv"
        
        # 실행 결과 파일이 없으면 패스
        if not os.path.exists(csv_path):
            continue
            
        # 실행 결과 파일이 존재하면 내용 검사
        try:
            df_result = pd.read_csv(csv_path)
            if df_result.empty:
                continue
                
            # 오답(False)이나 에러(NaN)가 하나라도 있는지 검사
            if (df_result['validation'] == False).any() or df_result['validation'].isnull().any():
                continue
            else:
                # 🌟 모두 True인 경우 (완벽한 정답 발견!)
                successful_orders[prob_id] = sol_order
                successful_codes[prob_id] = sol_code # [핵심] 실제 파이썬 코드 텍스트를 저장
                passed_problem_ids.add(prob_id)
                print(f"    🌟 Problem {prob_id}, Sol {sol_order}: 완벽 정답 확보! (이후 솔루션 무시)")
        except Exception:
            # 파일 읽기 에러 시 무시하고 넘어감
            pass

print(f"\n✅ 총 {len(passed_problem_ids)}개의 문제에 대한 정답 코드를 확보했습니다.")
print("\n🚀 [Step 2] problems.csv 병합 및 추출 (validated_problems.csv 생성)")

# --- Step 2: problems.csv 필터링 및 solution 열 추가 ---
chunk_iter_prob = pd.read_csv(PROBLEMS_FILE, chunksize=CHUNK_SIZE)
is_first_chunk_prob = True

for chunk_idx, df_prob_chunk in enumerate(chunk_iter_prob):
    print(f"  > [Prob] Chunk {chunk_idx + 1} 처리 중...")
    
    # 정답을 찾은(passed_problem_ids 안에 있는) 문제만 싹 필터링
    # 원본 파일에서 id 열이 문제 번호를 의미함
    filtered_prob = df_prob_chunk[df_prob_chunk['id'].isin(passed_problem_ids)].copy()
    
    if not filtered_prob.empty:
        # [핵심] 통과한 정답 코드를 새로운 'solution' 열에 매핑하여 삽입
        filtered_prob['solution'] = filtered_prob['id'].map(successful_codes)
        
        # 바로 최종 파일로 누적 저장 (중간 파일 불필요)
        mode = 'w' if is_first_chunk_prob else 'a'
        header = is_first_chunk_prob
        filtered_prob.to_csv(VALIDATED_PROB_FILE, mode=mode, header=header, index=False, encoding='utf-8-sig')
        is_first_chunk_prob = False

print("\n🚀 [Step 3] problem_testcases.csv 필터링 (validated_problem_testcases.csv 생성)")

# --- Step 3: problem_testcases.csv 필터링 및 validation 열 갱신 ---
chunk_iter_tc = pd.read_csv(TESTCASES_FILE, chunksize=CHUNK_SIZE)
is_first_chunk_tc = True

for chunk_idx, df_tc_chunk in enumerate(chunk_iter_tc):
    print(f"  > [TC] Chunk {chunk_idx + 1} 처리 중...")
    
    # 정답을 찾은 문제의 테스트케이스만 싹 필터링
    filtered_tc = df_tc_chunk[df_tc_chunk['problem_id'].isin(passed_problem_ids)].copy()
    
    if not filtered_tc.empty:
        # validation 열 값을 통과한 솔루션 번호(solution_order)로 변경
        filtered_tc['validation'] = filtered_tc['problem_id'].map(successful_orders)
        
        # 자료형 보장 (1.0 -> 1 로 정수형 변환)
        try:
            filtered_tc['validation'] = filtered_tc['validation'].astype(int)
        except:
            pass
            
        # 바로 최종 파일로 누적 저장 (중간 파일 불필요)
        mode = 'w' if is_first_chunk_tc else 'a'
        header = is_first_chunk_tc
        filtered_tc.to_csv(VALIDATED_TC_FILE, mode=mode, header=header, index=False, encoding='utf-8-sig')
        is_first_chunk_tc = False

print("\n🎉 모든 취합 및 데이터베이스 병합 작업이 완료되었습니다!")
print(f"최종 결과물 1: {VALIDATED_PROB_FILE} (정답 코드가 포함된 문제 목록)")
print(f"최종 결과물 2: {VALIDATED_TC_FILE} (검증 완료된 테스트케이스 목록)")

🚀 [Step 1] problem_solutions.csv 검사 시작 (정답 솔루션 코드 확보)
  > [Sol] Chunk 1 처리 중...
    🌟 Problem 2, Sol 1: 완벽 정답 확보! (이후 솔루션 무시)
    🌟 Problem 11, Sol 1: 완벽 정답 확보! (이후 솔루션 무시)
    🌟 Problem 19, Sol 1: 완벽 정답 확보! (이후 솔루션 무시)
    🌟 Problem 41, Sol 4: 완벽 정답 확보! (이후 솔루션 무시)
    🌟 Problem 51, Sol 1: 완벽 정답 확보! (이후 솔루션 무시)
    🌟 Problem 53, Sol 1: 완벽 정답 확보! (이후 솔루션 무시)
    🌟 Problem 58, Sol 1: 완벽 정답 확보! (이후 솔루션 무시)
    🌟 Problem 59, Sol 1: 완벽 정답 확보! (이후 솔루션 무시)
    🌟 Problem 64, Sol 1: 완벽 정답 확보! (이후 솔루션 무시)
    🌟 Problem 73, Sol 1: 완벽 정답 확보! (이후 솔루션 무시)
    🌟 Problem 78, Sol 1: 완벽 정답 확보! (이후 솔루션 무시)
    🌟 Problem 89, Sol 1: 완벽 정답 확보! (이후 솔루션 무시)
    🌟 Problem 92, Sol 1: 완벽 정답 확보! (이후 솔루션 무시)
    🌟 Problem 96, Sol 1: 완벽 정답 확보! (이후 솔루션 무시)
    🌟 Problem 100, Sol 1: 완벽 정답 확보! (이후 솔루션 무시)
    🌟 Problem 106, Sol 3: 완벽 정답 확보! (이후 솔루션 무시)
    🌟 Problem 113, Sol 1: 완벽 정답 확보! (이후 솔루션 무시)
    🌟 Problem 116, Sol 1: 완벽 정답 확보! (이후 솔루션 무시)
    🌟 Problem 128, Sol 1: 완벽 정답 확보! (이후 솔루션 무시)
    🌟 Problem 138, Sol 